# 4.3 Secondary outcomes - energy use by equipment

This notebook does the following:
    - Simple diff in diff analysis of secondary outcome (energy use by equipment type)

## Set-up

In [1]:
# Set-up
import pandas as pd
import numpy as np
import sys
import re
import importlib
from pathlib import Path
CODE_ROOT = Path.cwd().parents[1]
sys.path.append(str(CODE_ROOT))
import config
import os
import statsmodels.formula.api as smf
import pyfixest as pf
from make_regression_table import make_regression_table

In [2]:
# Load data
df = pd.read_csv(
    config.CLEAN_DATA / "final_dataset.csv", 
    keep_default_na=False, # Keep "None" as a string, not NaN
    na_values=[""] # Only treat empty strings as NaN
)

In [3]:
# Construct post variable
df["post"] = (df["survey"] == "EL").astype(int)

In [4]:
# Keep only labgroups that have both pre and post observations
labgroup_counts = df.groupby("labgroupid")["survey"].nunique()
labgroups_to_keep = labgroup_counts[labgroup_counts == 2].index
df = df[df["labgroupid"].isin(labgroups_to_keep)].copy()

## (1) Simple diff in diff regression


In [5]:
fit_levels = pf.feols(
    "annual_electricity_total ~ treated:post | labgroupid + post",
    data=df,
    vcov={"CRV1": "labgroupid"}
)

fit_levels.summary()

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


###

Estimation:  OLS
Dep. var.: annual_electricity_total, Fixed effects: labgroupid + post
sample: None = all
Inference:  CRV1
Observations:  218

| Coefficient   |   Estimate |   Std. Error |   t value |   Pr(>|t|) |     2.5% |   97.5% |
|:--------------|-----------:|-------------:|----------:|-----------:|---------:|--------:|
| treated:post  |    -19.563 |      111.782 |    -0.175 |      0.861 | -241.136 | 202.009 |
---
RMSE: 288.413 R2: 1.0 R2 Within: 0.0 


In [6]:
df['log_electricity'] = np.log1p(df['annual_electricity_total']) 

fit_log = pf.feols(
    "log_electricity ~ treated:post | labgroupid + post",
    data=df,
    vcov={"CRV1": "labgroupid"}
)
fit_log.summary()

###

Estimation:  OLS
Dep. var.: log_electricity, Fixed effects: labgroupid + post
sample: None = all
Inference:  CRV1
Observations:  218

| Coefficient   |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:--------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| treated:post  |     -0.053 |        0.051 |    -1.038 |      0.302 | -0.153 |   0.048 |
---
RMSE: 0.127 R2: 0.994 R2 Within: 0.011 


## (2) Heterogeneity analysis

In [7]:
# Equipment energy analyses — store results by equipment type
outcomes = {
    "Fume cupboard":  "annual_electricity_fc",
    "Fridge":         "annual_electricity_fridge",
    "Freezer":        "annual_electricity_freezer",
    "ULT":            "annual_electricity_ult",
    "Cryostat":       "annual_electricity_cryostat",
    "MSC":            "annual_electricity_microbio",
    "Incubator":      "annual_electricity_incubator",
    "Glassware":      "annual_electricity_glassware",
    "Bath":           "annual_electricity_bath",
    "Heater":         "annual_electricity_heater",
    "IT":             "annual_electricity_it",
}

# Pre-create log columns
for col in outcomes.values():
    df[f"log_{col}"] = np.log1p(df[col])

fit_levels_outcomes = {}
fit_log_outcomes    = {}

for name, col in outcomes.items():
    fit_levels_outcomes[name] = pf.feols(
        f"{col} ~ treated:post | labgroupid + post",
        data=df, vcov={"CRV1": "labgroupid"}
    )
    fit_log_outcomes[name] = pf.feols(
        f"log_{col} ~ treated:post | labgroupid + post",
        data=df, vcov={"CRV1": "labgroupid"}
    )

/Users/drutna/miniconda3/envs/labrct/lib/python3.10/site-packages/pyfixest/estimation/models/_result_accessor_mixin.py:86: RuntimeWarning: invalid value encountered in divide
  self._tstat = self._beta_hat / self._se
/Users/drutna/miniconda3/envs/labrct/lib/python3.10/site-packages/pyfixest/estimation/models/_result_accessor_mixin.py:139: RuntimeWarning: invalid value encountered in scalar divide
  self._r2_within = 1 - (ssu / ssy_within)
/Users/drutna/miniconda3/envs/labrct/lib/python3.10/site-packages/pyfixest/estimation/models/_result_accessor_mixin.py:140: RuntimeWarning: invalid value encountered in scalar divide
  self._adj_r2_within = 1 - (ssu / ssy_within) * adj_factor_within
/Users/drutna/miniconda3/envs/labrct/lib/python3.10/site-packages/pyfixest/estimation/models/_result_accessor_mixin.py:86: RuntimeWarning: invalid value encountered in divide
  self._tstat = self._beta_hat / self._se
/Users/drutna/miniconda3/envs/labrct/lib/python3.10/site-packages/pyfixest/estimation/mode

In [8]:
# Export to nice table
table = make_regression_table(
    fit_list = [
        fit_levels,
        fit_levels_outcomes["Fume cupboard"], fit_levels_outcomes["Fridge"], fit_levels_outcomes["Freezer"],
        fit_levels_outcomes["ULT"], fit_levels_outcomes["Cryostat"], fit_levels_outcomes["MSC"],
        fit_levels_outcomes["Incubator"], fit_levels_outcomes["Glassware"], fit_levels_outcomes["Bath"],
        fit_levels_outcomes["Heater"], fit_levels_outcomes["IT"],
    ],
    model_names   = ["(1)", "(2)", "(3)", "(4)", "(5)", "(6)", "(7)", "(8)", "(9)", "(10)", "(11)", "(12)"],
    keep_vars     = ["treated:post"],
    var_labels    = {"treated:post": "Treated $\\times$ Post"},
    fe_rows       = {
        "Lab group FE": [True] * 12,
        "Time FE":      [True] * 12,
    },
    col_subgroups = {
        "Total":         [0],
        "Fume cupboard": [1],
        "Fridge":        [2],
        "Freezer":       [3],
        "ULT":           [4],
        "Cryostat":      [5],
        "MSC":           [6],
        "Incubator":     [7],
        "Glassware":     [8],
        "Bath":          [9],
        "Heater":        [10],
        "IT":            [11],
    },
    baseline_mean = "auto",
    decimals      = [1] * 12,
    mean_decimals = 0,
    r2_type       = None,
    col1_width    = "4cm",
    coln_width    = "1.5cm",
)
table_path = config.OUTPUT / "5_Regression_Tables" / "secondary_equipment_levels.tex"
_ = table_path.write_text(table)

/Users/drutna/miniconda3/envs/labrct/lib/python3.10/site-packages/pyfixest/estimation/models/_result_accessor_mixin.py:86: RuntimeWarning: invalid value encountered in divide
  self._tstat = self._beta_hat / self._se


In [9]:
# Export to nice table
table = make_regression_table(
    fit_list = [
        fit_log,
        fit_log_outcomes["Fume cupboard"], fit_log_outcomes["Fridge"], fit_log_outcomes["Freezer"],
        fit_log_outcomes["ULT"], fit_log_outcomes["Cryostat"], fit_log_outcomes["MSC"],
        fit_log_outcomes["Incubator"], fit_log_outcomes["Glassware"], fit_log_outcomes["Bath"],
        fit_log_outcomes["Heater"], fit_log_outcomes["IT"],
    ],
    model_names   = ["(1)", "(2)", "(3)", "(4)", "(5)", "(6)", "(7)", "(8)", "(9)", "(10)", "(11)", "(12)"],
    keep_vars     = ["treated:post"],
    var_labels    = {"treated:post": "Treated $\\times$ Post"},
    fe_rows       = {
        "Lab group FE": [True] * 12,
        "Time FE":      [True] * 12,
    },
    col_subgroups = {
        "Total":         [0],
        "Fume cupboard": [1],
        "Fridge":        [2],
        "Freezer":       [3],
        "ULT":           [4],
        "Cryostat":      [5],
        "MSC":           [6],
        "Incubator":     [7],
        "Glassware":     [8],
        "Bath":          [9],
        "Heater":        [10],
        "IT":            [11],
    },
    baseline_mean = "auto",
    decimals      = [3] * 12,
    mean_decimals = 3,
    r2_type       = None,
    col1_width    = "4cm",
    coln_width    = "1.5cm",
)
table_path = config.OUTPUT / "5_Regression_Tables" / "secondary_equipment_log.tex"
_ = table_path.write_text(table)